# Build the macro-release shock inputs

Combines the Bloomberg ECO EZ calendar exports (`BB_calender.zip`) with the daily 2y €STR OIS series (`ois_2y.csv`, ECB SDW) into **`Data/release_shocks.csv`** — a daily wide panel with one standardised-surprise column per release type, used to estimate the first-stage news regression in Stata (γ per release type; fitted value = daily macro shock in bp). Also writes **`Data/release_types_meta.csv`** (one row per release type: σ, relevance, obs counts) to select regressors by relevance rule.

Design decisions (see `so_what_plan.md`):
- splice: old export up to end-2022, new export from 2023 (2023 overlap cross-validated below);
- surprise = (actual − survey median) / time-series σ of that release type, σ estimated on the full 2000–2025 history (SW/AGM convention — **not** Bloomberg's forecaster-dispersion-scaled surprise column);
- flash/final/advance variants (Period suffix P/F/A/S/T) are separate release types;
- ECB rate-decision rows are dropped from the release set; ECB decision days are flagged (`ecb_day`) so the first stage can exclude them.

In [1]:
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()                  # repo root; adjust if running elsewhere
DATA_OUT = ROOT / 'Data'
DATA_OUT.mkdir(exist_ok=True)

CAL_DIR = ROOT / 'BB_calender'
if not CAL_DIR.exists():           # auto-extract the committed zip
    with zipfile.ZipFile(ROOT / 'BB_calender.zip') as z:
        z.extractall(ROOT)
OLD_XLSX = CAL_DIR / 'BB_00_23.xlsx'
NEW_XLSX = CAL_DIR / 'BB_23_25.xlsx'
OIS_CSV  = ROOT / 'ois_2y.csv'

SPLICE = pd.Timestamp('2023-01-01')   # old file strictly before, new file from here
ECB_TICKERS = {'EURR002W Index', 'EUORDEPO Index', 'EUORMARG Index'}
MIN_OBS_SIGMA = 20                    # below this, fall back to ticker-level sigma

## Load and normalise the two calendar exports

In [2]:
old = pd.read_excel(OLD_XLSX)
old['date'] = pd.to_datetime(old['release_date'])
old_n = old.rename(columns={'Ticker': 'ticker', 'Event': 'event', 'Period': 'period',
                            'num_Actual': 'actual', 'num_Surv_M': 'median'})
old_n = old_n[['date', 'ticker', 'event', 'period', 'actual', 'median']]

new = pd.read_excel(NEW_XLSX)
new['date'] = pd.to_datetime(new['Datum/Zeit'], errors='coerce').dt.normalize()
new = new[new['date'].notna()].copy()                     # drops the download-stamp footer
for c in ['Ist', 'Umfr.(Md)', 'S']:
    new[c] = pd.to_numeric(new[c], errors='coerce')
new_n = new.rename(columns={'Ticker': 'ticker', 'Ereignis': 'event', 'Periode': 'period',
                            'Ist': 'actual', 'Umfr.(Md)': 'median'})
new_n = new_n[['date', 'ticker', 'event', 'period', 'actual', 'median']]

# cross-validate the 2023 overlap before splicing
o23 = old_n[old_n['date'].dt.year == 2023]
n23 = new_n[new_n['date'].dt.year == 2023]
m = o23.merge(n23, on=['date', 'ticker'], suffixes=('_o', '_n')).dropna(subset=['actual_o', 'actual_n'])
match = np.isclose(m['actual_o'], m['actual_n'], rtol=1e-3).mean()
print(f'2023 overlap: {len(m)} ticker-dates, actuals match {match:.1%}')
assert match > 0.99, 'overlap mismatch - check file versions before splicing'

comb = pd.concat([old_n[old_n['date'] < SPLICE], new_n[new_n['date'] >= SPLICE]], ignore_index=True)
print(f'spliced calendar: {len(comb)} rows, {comb["date"].min().date()} -> {comb["date"].max().date()}')
CAL_END = comb['date'].max()          # panel must not extend past calendar coverage

2023 overlap: 413 ticker-dates, actuals match 100.0%
spliced calendar: 13303 rows, 2000-01-04 -> 2025-10-23


## ECB decision days

The calendar contains the ECB rate decisions themselves. These days are the paper's monetary-policy events (EA-MPD); the decision rows are dropped from the release set and the days flagged, so the Stata first stage can exclude them. Cross-check the flagged dates against the EA-MPD event dates when merging into the bond panel.

In [3]:
ecb_days = set(comb.loc[comb['ticker'].isin(ECB_TICKERS), 'date'])
comb = comb[~comb['ticker'].isin(ECB_TICKERS)].copy()
print(f'{len(ecb_days)} ECB decision days flagged (2000-2025); release rows remaining: {len(comb)}')

288 ECB decision days flagged (2000-2025); release rows remaining: 12804


## Release types and standardised surprises

Release type = ticker + Period variant (P/F/A/S/T). Raw surprise = actual − median; standardised by the type's full-history time-series σ (ticker-level fallback when a variant has fewer than 20 observations).

**Flash PMI reconstruction.** In both exports the flash PMI rows (variant P) carry the pre-release survey median but a *blank actual* — across the entire history — so the flash print, which is the market-moving PMI release, would be lost. The flash actual is recoverable from inside the file: the forecaster consensus for the *final* print (F row of the same reference month) is anchored on the published flash (F-row forecaster ranges sit within ±0.1 of one value). We therefore set flash actual := same-reference-month F-row median. No look-ahead: the flash print is public on the P-row date; only its numerical value is being recovered from a later-dated field. → Cross-check with the colleague whether the ECO export can deliver flash actuals directly.

In [4]:
comb['base'] = comb['ticker'].str.replace(' Index', '', regex=False).str.strip()
comb['variant'] = comb['period'].astype(str).str.extract(r'\b([APFST])$')[0].fillna('')
comb['rtype'] = np.where(comb['variant'] == '', comb['base'], comb['base'] + '_' + comb['variant'])

# ---- recover flash PMI actuals (blank in the exports) from the final-print consensus ----
tok = comb['period'].astype(str).str.replace(r'\s*[APFST]$', '', regex=True).str.strip()
ref_m = pd.to_datetime(tok, format='%b', errors='coerce').dt.month
comb['ref_y'] = comb['date'].dt.year - (ref_m > comb['date'].dt.month).astype(int)
comb['ref_m'] = ref_m

is_pmi = comb['base'].str.startswith('MPMIEZ')
fmed = (comb[is_pmi & (comb['variant'] == 'F')].dropna(subset=['median', 'ref_m'])
        .set_index(['base', 'ref_y', 'ref_m'])['median'])
fmed = fmed[~fmed.index.duplicated()]
need = is_pmi & (comb['variant'] == 'P') & comb['actual'].isna() & comb['median'].notna() & comb['ref_m'].notna()
key = pd.MultiIndex.from_arrays([comb.loc[need, 'base'], comb.loc[need, 'ref_y'], comb.loc[need, 'ref_m']])
comb.loc[need, 'actual'] = fmed.reindex(key).to_numpy()
rec = comb.loc[need, 'actual'].notna().sum()
print(f'flash PMI actuals reconstructed from final-print consensus: {rec} of {need.sum()} P rows')

surp = comb.dropna(subset=['actual', 'median']).copy()
surp['raw'] = surp['actual'] - surp['median']

sig_type = surp.groupby('rtype')['raw'].agg(sigma='std', n_full='count')
sig_base = surp.groupby('base')['raw'].std().rename('sigma_base')
surp = surp.join(sig_type, on='rtype').join(sig_base, on='base')
surp['sigma_used'] = np.where(surp['n_full'] >= MIN_OBS_SIGMA, surp['sigma'], surp['sigma_base'])
surp = surp[surp['sigma_used'] > 0].copy()
surp['s_std'] = surp['raw'] / surp['sigma_used']

# relevance score (Bloomberg alert share) by ticker, from the 2023-25 export
rel = new.copy()
rel['base'] = rel['Ticker'].str.replace(' Index', '', regex=False).str.strip()
relevance = rel.groupby('base')['S'].max()
surp['relevance'] = surp['base'].map(relevance)

# same-day duplicates within a type (e.g. two reference periods released together): sum the news
dups = surp.groupby(['date', 'rtype']).size()
print(f'same-day duplicate type observations: {(dups > 1).sum()}')
surp_d = (surp.groupby(['date', 'rtype'], as_index=False)
              .agg(s_std=('s_std', 'sum'), base=('base', 'first'), variant=('variant', 'first'),
                   sigma_used=('sigma_used', 'first'), n_full=('n_full', 'first'),
                   relevance=('relevance', 'first')))
in_win = surp_d[surp_d['date'] >= '2021-01-01']
print(f'full history: {len(surp_d)} obs, {surp_d["rtype"].nunique()} types | 2021+: '
      f'{len(in_win)} obs on {in_win["date"].nunique()} dates')

flash PMI actuals reconstructed from final-print consensus: 658 of 660 P rows
same-day duplicate type observations: 0
full history: 7794 obs, 79 types | 2021+: 1637 obs on 643 dates


## Daily OIS series (ECB SDW export)

In [5]:
ois = pd.read_csv(OIS_CSV, header=None, usecols=[0, 1], names=['date', 'ois_2y'],
                  engine='python', skiprows=5)
ois['date'] = pd.to_datetime(ois['date'], errors='coerce')
ois['ois_2y'] = pd.to_numeric(ois['ois_2y'], errors='coerce')
ois = ois.dropna().sort_values('date').reset_index(drop=True)
ois['d_ois2y_bp'] = ois['ois_2y'].diff() * 100          # percent p.a. -> basis points
print(f'OIS: {len(ois)} days, {ois["date"].min().date()} -> {ois["date"].max().date()}')
print(f'daily change (bp): sd={ois["d_ois2y_bp"].std():.2f}, '
      f'p1={ois["d_ois2y_bp"].quantile(.01):.1f}, p99={ois["d_ois2y_bp"].quantile(.99):.1f}')

OIS: 1430 days, 2021-01-04 -> 2026-07-27
daily change (bp): sd=5.11, p1=-15.2, p99=14.4


## Wide daily panel

Trading days are the OIS dates. Releases on non-trading days roll forward to the next trading day (the news is priced at the next open). Panel is truncated at the calendar's last date so trailing days are not misread as news-free. Non-release days carry 0 in every surprise column (AGM: newsᵢ = 0 when variable i is not released).

In [6]:
trading = ois['date'].to_numpy()

def roll_fwd(dates):
    idx = np.searchsorted(trading, dates.to_numpy(), side='left')
    ok = idx < len(trading)
    return pd.Series(np.where(ok, trading[np.clip(idx, 0, len(trading) - 1)], np.datetime64('NaT')),
                     index=dates.index)

panel_src = surp_d[surp_d['date'] >= '2021-01-01'].copy()
panel_src['tday'] = roll_fwd(panel_src['date'])
rolled = (panel_src['tday'] != panel_src['date']).sum()
panel_src = panel_src.dropna(subset=['tday'])
print(f'release obs rolled to next trading day: {rolled}; dropped beyond OIS range: '
      f'{(surp_d["date"] >= "2021-01-01").sum() - len(panel_src)}')

wide = (panel_src.pivot_table(index='tday', columns='rtype', values='s_std', aggfunc='sum')
        .rename(columns=lambda c: 's_' + c.lower()))
n_rel = panel_src.groupby('tday').size().rename('n_releases')

panel = ois.set_index('date').join(wide).join(n_rel)
panel = panel[panel.index <= CAL_END]
s_cols = [c for c in panel.columns if c.startswith('s_')]
panel[s_cols] = panel[s_cols].fillna(0.0)
panel['n_releases'] = panel['n_releases'].fillna(0).astype(int)

ecb_rolled = roll_fwd(pd.Series(sorted(d for d in ecb_days if d >= pd.Timestamp('2021-01-01')))).dropna()
panel['ecb_day'] = panel.index.isin(set(ecb_rolled)).astype(int)

dead = [c for c in s_cols if (panel[c] == 0).all()]
panel = panel.drop(columns=dead)
s_cols = [c for c in s_cols if c not in dead]
print(f'dropped all-zero columns (types not observed in-window): {len(dead)}')
print(f'panel: {len(panel)} trading days, {len(s_cols)} release types, '
      f'{(panel["n_releases"] > 0).sum()} release days, {panel["ecb_day"].sum()} ECB days')

release obs rolled to next trading day: 0; dropped beyond OIS range: 0
dropped all-zero columns (types not observed in-window): 2
panel: 1238 trading days, 41 release types, 643 release days, 38 ECB days


## Sanity checks (preview only — the official first stage runs in Stata)

In [7]:
# largest |dOIS| days: are they ECB days or big-surprise release days?
top = panel.reindex(panel['d_ois2y_bp'].abs().sort_values(ascending=False).head(12).index)
top_s = top[s_cols].apply(lambda r: '' if r.abs().max() == 0
                          else f'{r.abs().idxmax()[2:]}={r[r.abs().idxmax()]:.1f}', axis=1)
print(pd.DataFrame({'d_ois_bp': top['d_ois2y_bp'].round(1), 'ecb': top['ecb_day'],
                    'n_rel': top['n_releases'], 'largest surprise': top_s}).to_string())

# naive OLS preview of the first stage: relevance>=50, at least 8 release days, ECB days excluded
meta_rel = panel_src.groupby('rtype')['relevance'].first()
keep = ['s_' + t.lower() for t, r in meta_rel.items()
        if r >= 50 and 's_' + t.lower() in s_cols and (panel['s_' + t.lower()] != 0).sum() >= 8]
sub = panel[(panel['ecb_day'] == 0) & panel['d_ois2y_bp'].notna()]
X = np.column_stack([np.ones(len(sub)), sub[keep].to_numpy()])
gamma = np.linalg.lstsq(X, sub['d_ois2y_bp'].to_numpy(), rcond=None)[0]
prev = pd.DataFrame({'gamma_bp_per_sigma': gamma[1:].round(2),
                     'n_days': [(sub[c] != 0).sum() for c in keep]}, index=keep)
print('\npreview first stage (constant excluded from display):')
print(prev.sort_values('gamma_bp_per_sigma', key=abs, ascending=False).to_string())

            d_ois_bp  ecb  n_rel largest surprise
date                                             
2023-03-13     -32.9    0      0                 
2023-03-15     -27.7    0      2     euitemum=0.7
2023-03-14      25.3    0      0                 
2022-10-28      23.5    0      3     euscemu=-0.6
2022-07-22     -23.2    0      3  mpmiezma_p=-1.1
2022-06-23     -22.8    0      3  mpmiezsa_p=-1.6
2022-10-27     -22.3    1      0                 
2022-06-15     -22.0    0      3      xtsbez=-5.2
2023-03-17     -22.0    0      3                 
2022-12-15      21.4    1      0                 
2023-03-21      20.3    0      0                 
2022-06-13      19.7    0      0                 



preview first stage (constant excluded from display):
              gamma_bp_per_sigma  n_days
s_eugnemuq_a                5.70      17
s_eugnemuy_a               -3.80      14
s_mpmiezsa_p                1.82      52
s_cpexemuy_p                1.80      38
s_mpmiezca_p                1.69      53
s_eccpemum_p               -1.66      40
s_mpmiezma_p                1.26      55
s_eccpemuy_f               -1.12      10
s_umrtemu                   0.98      29
s_eccpest                   0.87      36
s_mpmiezma_f               -0.85      41
s_eccpemum_f               -0.82      10
s_euitemum                 -0.79      50
s_rssaemum                 -0.45      47
s_mpmiezsa_f               -0.42      45
s_eugnemuq_f                0.24      10
s_rswaemuy                 -0.20      53
s_euccemu_p                -0.14      50
s_euppemuy                  0.13      44
s_mpmiezca_f                0.07      42
s_ecmam3yy                 -0.05      51
s_eugnemuy_f                0.05      12
s_

## Export

In [8]:
out = panel.reset_index().rename(columns={'index': 'date', 'tday': 'date'})
out['date'] = pd.to_datetime(out['date']).dt.strftime('%Y-%m-%d')
front = ['date', 'ois_2y', 'd_ois2y_bp', 'ecb_day', 'n_releases']
out = out[front + sorted(c for c in out.columns if c.startswith('s_'))]
out.to_csv(DATA_OUT / 'release_shocks.csv', index=False)

names = comb.groupby('base')['event'].agg(lambda s: s.value_counts().index[0])
meta = (panel_src.groupby('rtype')
        .agg(base=('base', 'first'), variant=('variant', 'first'), sigma=('sigma_used', 'first'),
             n_full_hist=('n_full', 'first'), relevance=('relevance', 'first'),
             n_days_2021plus=('tday', 'nunique'))
        .reset_index())
meta['column'] = 's_' + meta['rtype'].str.lower()
meta['event'] = meta['base'].map(names)
meta = meta[meta['column'].isin(out.columns)]
meta = meta[['column', 'rtype', 'base', 'variant', 'event', 'relevance', 'sigma',
             'n_full_hist', 'n_days_2021plus']].sort_values('relevance', ascending=False)
meta.to_csv(DATA_OUT / 'release_types_meta.csv', index=False)

print(f'wrote {DATA_OUT / "release_shocks.csv"}  ({out.shape[0]} rows x {out.shape[1]} cols)')
print(f'wrote {DATA_OUT / "release_types_meta.csv"}  ({len(meta)} release types)')
meta.head(20)

wrote /home/user/JMP_current/Data/release_shocks.csv  (1238 rows x 46 cols)
wrote /home/user/JMP_current/Data/release_types_meta.csv  (41 release types)


,column,rtype,base,variant,event,relevance,sigma,n_full_hist,n_days_2021plus
4,s_eccpemuy_f,ECCPEMUY_F,ECCPEMUY,F,CPI YoY,95.2381,0.000445,181,57
14,s_eugnemuq_f,EUGNEMUQ_F,EUGNEMUQ,F,GDP SA QoQ,90.4762,0.000886,74,17
15,s_eugnemuq_p,EUGNEMUQ_P,EUGNEMUQ,P,GDP SA QoQ,90.4762,0.000394,96,17
16,s_eugnemuq_s,EUGNEMUQ_S,EUGNEMUQ,S,GDP SA QoQ,90.4762,0.002148,2,2
17,s_eugnemuq_t,EUGNEMUQ_T,EUGNEMUQ,T,GDP SA QoQ,90.4762,0.002148,2,2
13,s_eugnemuq_a,EUGNEMUQ_A,EUGNEMUQ,A,GDP SA QoQ,90.4762,0.003478,96,19
33,s_mpmiezma_f,MPMIEZMA_F,MPMIEZMA,F,HCOB Eurozone Manufacturing PMI,90.0000,0.207128,51,51
34,s_mpmiezma_p,MPMIEZMA_P,MPMIEZMA,P,HCOB Eurozone Manufacturing PMI,90.0000,1.223413,220,57
20,s_eugnemuy_p,EUGNEMUY_P,EUGNEMUY,P,GDP SA YoY,88.0952,0.000802,94,17
22,s_eugnemuy_t,EUGNEMUY_T,EUGNEMUY,T,GDP SA YoY,88.0952,0.002120,2,2


## Stata: first stage

```stata
import delimited using "Data/release_shocks.csv", clear case(preserve)
gen bdate = date(date, "YMD")
format bdate %td

* ---- First stage (AGM eq. 1 structure), ECB monetary event days excluded ----
* Primary rule: relevance >= 50. Column list in Data/release_types_meta.csv;
* robustness: thresholds 60/70 (drop columns), or all columns (s_*).
reg d_ois2y_bp s_* if ecb_day == 0, vce(robust)

* Daily macro shock (bp) = fitted news component, net of the constant
predict shock_macro if ecb_day == 0, xb
replace shock_macro = shock_macro - _b[_cons] if !missing(shock_macro)
replace shock_macro = 0 if n_releases == 0 & ecb_day == 0
```

Then merge `shock_macro` into the bond panel by date and use it as the drop-in
replacement for the MP surprise (interaction, size terciles, per-event betas).
ECB days keep the EA-MPD shock — the two laboratories stay separate. The 38
flagged `ecb_day`s in-window should line up with the paper's 38 EA-MPD events
(cross-check when merging).

**Regressor-set guidance.**
- Restrict to types with a reasonable number of in-window release days
  (`n_days_2021plus` in the meta file, e.g. ≥ 10) — the GDP second/third-release
  variants have 1–2 observations and only add noise.
- The flash-CPI bundle (headline estimate `s_eccpest`, core `s_cpexemuy_p`,
  MoM `s_eccpemum_p`) is released simultaneously with same-day surprise
  correlations up to 0.96; the GDP QoQ/YoY pair is the same case. Individual γs
  inside a bundle are therefore not separately interpretable (signs can flip);
  the **fitted value is unaffected**. For a clean γ table in the paper, keep one
  print per bundle (drop the CPI MoM and GDP YoY duplicates) and report
  bundle-level joint F-tests.